Vamos a tantear diferentes modelos y configuraciones antes de parametrizar el definitivo. Gracias a la libreria lazypredict hara un tanteo en los principales modelos de regresion y clasificacion.



In [3]:
import pandas as pd
import numpy as np, random
random.seed(42)

In [4]:
df2 = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/presplit.csv')

In [ ]:
df2.info()

In [ ]:
import pandas as pd

def redondear_columnas(df, columnas):

    for columna in columnas:
        if columna in df.columns:
            df[columna] = df[columna].apply(lambda x: round(x, 1) if pd.notnull(x) else x)
        else:
            print(f"La columna '{columna}' no existe en el DataFrame.")
    return df

columnas_a_redondear = ['confianza_promedio', 'satisf_media', 'ppl']

# Aplicar el redondeo a las columnas especificadas
df2= redondear_columnas(df2, columnas_a_redondear)

# Verificar los valores redondeados
print(df2.head(4))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
import os


def bestclass(df, target_column, feature_range, exclude_columns=None):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    classifiers = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_classifier_name = ""

    for k in range(feature_range[0], feature_range[1] + 1):
        for classifier_name, classifier in classifiers.items():
            selector = SelectKBest(score_func=f_classif, k=k)
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

            classifier.fit(X_train_selected, y_train)
            y_pred_prob = classifier.predict_proba(X_test_selected)

            auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')

            if auc > best_auc:
                best_auc = auc
                best_model = classifier
                best_k = k
                best_features = X.columns[selector.get_support()].tolist()
                best_classifier_name = classifier_name

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'bestclassifier_iterative_corrected.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_classifier_name}\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_classifier_name


exclude_cols = ['lrscale', 'lrscale2_fc','cntry','lw_pnd']  
best_model, best_k, best_features, best_classifier_name = bestclass(df2, 'lrscale_fc', (10, 18), exclude_columns=exclude_cols)

print(f"Mejor modelo: {best_classifier_name}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")

Es increible que ha dado un 0.67 de score con SVM , cuando svm clasifica casi obligatoriamente con datos escalados.
Vamos a escalar las variables numericas no factorizadas.
E hiperparametrizamos.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
#probamos con la variante lrscale de 5 opciones factorizadas
def best_svm_scaled(df, target_column, feature_range, scale_columns=None, exclude_columns=None):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    if scale_columns:
        
        scaler_minmax = MinMaxScaler()
        X_minmax = X.copy()
        X_minmax[scale_columns] = scaler_minmax.fit_transform(X_minmax[scale_columns])

        
        scaler_std = StandardScaler()
        X_std = X.copy()
        X_std[scale_columns] = scaler_std.fit_transform(X_std[scale_columns])

       
        X_combined = pd.concat([X, X_minmax[scale_columns].add_suffix('_minmax'), X_std[scale_columns].add_suffix('_std')], axis=1)
    else:
        X_combined = X.copy()

    X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_accuracy = 0
    best_precision = 0
    best_recall = 0
    best_f1 = 0

    for k in range(feature_range[0], feature_range[1] + 1):
        selector = SelectKBest(score_func=f_classif, k=k)
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)

        classifier = SVC(probability=True, random_state=42)
        classifier.fit(X_train_selected, y_train)
        y_pred_prob = classifier.predict_proba(X_test_selected)
        y_pred = classifier.predict(X_test_selected)

        auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        if auc > best_auc:
            best_auc = auc
            best_model = classifier
            best_k = k
            best_features = X_combined.columns[selector.get_support()].tolist()
            best_accuracy = accuracy
            best_precision = precision
            best_recall = recall
            best_f1 = f1

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'BestSVMdeflr4_scaled.txt'), 'w') as f:
        f.write(f"Mejor modelo: SVM\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Accuracy: {best_accuracy}\n")
        f.write(f"Precision: {best_precision}\n")
        f.write(f"Recall: {best_recall}\n")
        f.write(f"F1-score: {best_f1}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1


exclude_cols = ['lrscale', 'lrscale2_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']

best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_scaled(df2, 'lrscale_fc', (10, 20), scale_columns=scale_cols, exclude_columns=exclude_cols)

print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

In [ ]:
#probamos con la variante lrscale de 5 opciones factorizadas
exclude_cols = ['lrscale', 'lrscale_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']
best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_scaled(df2, 'lrscale2_fc', (10, 20), scale_columns=scale_cols, exclude_columns=exclude_cols)
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
#Probamos con criterio chi2


def best_svm_chi2_minmax(df, target_column, feature_range, scale_columns=None, exclude_columns=None, transform_negative=False):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    # Inicializar X_combined antes del condicional
    X_combined = X.copy()

    if scale_columns:
        # Generar variables escaladas con MinMaxScaler
        scaler_minmax = MinMaxScaler()
        X_minmax = X.copy()
        X_minmax[scale_columns] = scaler_minmax.fit_transform(X_minmax[scale_columns])

        # Concatenar las variables originales y escaladas
        X_combined = pd.concat([X, X_minmax[scale_columns].add_suffix('_minmax')], axis=1)

    X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_accuracy = 0
    best_precision = 0
    best_recall = 0
    best_f1 = 0

    for k in range(feature_range[0], feature_range[1] + 1):
        selector = SelectKBest(score_func=chi2, k=k)

        if transform_negative:
            X_train_selected = selector.fit_transform(X_train - X_train.min(), y_train)
            X_test_selected = selector.transform(X_test - X_test.min())
        else:
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

        classifier = SVC(probability=True, random_state=42)
        classifier.fit(X_train_selected, y_train)
        y_pred_prob = classifier.predict_proba(X_test_selected)
        y_pred = classifier.predict(X_test_selected)

        auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        if auc > best_auc:
            best_auc = auc
            best_model = classifier
            best_k = k
            best_features = X_combined.columns[selector.get_support()].tolist()
            best_accuracy = accuracy
            best_precision = precision
            best_recall = recall
            best_f1 = f1

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'BestSVMdeflr4_chi2_minmax.txt'), 'w') as f:
        f.write(f"Mejor modelo: SVM\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Accuracy: {best_accuracy}\n")
        f.write(f"Precision: {best_precision}\n")
        f.write(f"Recall: {best_recall}\n")
        f.write(f"F1-score: {best_f1}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1


exclude_cols = ['lrscale', 'lrscale2_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']

best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_chi2_minmax(df2, 'lrscale_fc', (7, 20), scale_columns=scale_cols, exclude_columns=exclude_cols, transform_negative=True)

print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

In [ ]:
exclude_cols = ['lrscale', 'lrscale_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']
best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_chi2_minmax(df2, 'lrscale2_fc', (7, 20), scale_columns=scale_cols, exclude_columns=exclude_cols, transform_negative=True)
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

Todos las combinaciones probadas estan en '.\4GA.DataScience\models\Allmodelsresumedefault.txt'

In [3]:

import pandas as pd
from sklearn.model_selection import train_test_split

df2 = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/presplit.csv')
desired_size = 2560
proportion = desired_size / len(df2)
df2strat, _ = train_test_split(df2, train_size=proportion, stratify=df2['cntry'], random_state=42)

numeric_cols = df2strat.select_dtypes(include=['number']).columns
df2strat[numeric_cols] = df2strat[numeric_cols].round(1)
for col in df2strat.columns:
    df2strat[col] = df2strat[col].fillna(df2strat[col].mode()[0])

print(f"Tamaño del DataFrame estratificado: {len(df2strat)}")

Tamaño del DataFrame estratificado: 2560


In [ ]:
import pandas as pd
import os
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import SVC
from sklearn.metrics import cohen_kappa_score, accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.preprocessing import MinMaxScaler
#hiperparametrizamos alrededor de este modelo base.
'''Mejor modelo: SVM
variable lrscale 4fc
chi scale minmax.
AUC: 0.6655128495412039
Accuracy: 0.544921875
Precision: 0.559275503676969
Recall: 0.544921875
F1-score: 0.411087474810554
N�mero de caracter�sticas: 16
Caracter�sticas: ['ppltrst', 'polintr', 'stfdem', 'stfeco', 'stfgov', 'trstep', 'trstlgl', 'trstplt', 'trstprl', 'imbgeco', 'imwbcnt', 'rlgdgr', 'cntgrp_fc', 'cnt_fc', 'rel3fc', 'satisf_media']'''

def best_svm_chi2_minmax_hyper(df, target_column, feature_range, scale_columns=None, exclude_columns=None, transform_negative=False):

    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    X_combined = X.copy()

    if scale_columns:
        scaler_minmax = MinMaxScaler()
        X_minmax = X.copy()
        X_minmax[scale_columns] = scaler_minmax.fit_transform(X[scale_columns])
        X_combined = pd.concat([X, X_minmax[scale_columns].add_suffix('_minmax')], axis=1)

    X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

    best_model = None
    best_kappa = -float('inf')
    best_k = 0
    best_features = []
    best_accuracy = 0
    best_precision = 0
    best_recall = 0
    best_f1 = 0

    for k in range(feature_range[0], feature_range[1] + 1):
        selector = SelectKBest(score_func=chi2, k=k)

        if transform_negative:
            X_train_selected = selector.fit_transform(X_train - X_train.min(), y_train)
            X_test_selected = selector.transform(X_test - X_test.min())
        else:
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

        param_grid = {
            'C': [1, 10, 100],
            'kernel': ['rbf', 'poly'],
            'gamma': ['scale', 0.1]
        }

        kappa_scorer = make_scorer(cohen_kappa_score) #Crear un scorer personalizado

        grid_search = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=2, scoring=kappa_scorer, verbose=1) # Usar kappa_scorer
        grid_search.fit(X_train_selected, y_train)

        classifier = grid_search.best_estimator_
        y_pred = classifier.predict(X_test_selected)

        kappa = cohen_kappa_score(y_test, y_pred)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        if kappa > best_kappa:
            best_kappa = kappa
            best_model = classifier
            best_k = k
            best_features = X_combined.columns[selector.get_support()].tolist()
            best_accuracy = accuracy
            best_precision = precision
            best_recall = recall
            best_f1 = f1

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/models'
    os.makedirs(output_dir, exist_ok=True)

    model_filename = os.path.join(output_dir, 'best_svm_model_kappa.pkl')
    with open(model_filename, 'wb') as f:
        pickle.dump(best_model, f)

    with open(os.path.join(output_dir, 'BestSVMdeflr4_chi2_minmax_hyper_kappa.txt'), 'w') as f:
        f.write(f"Mejor modelo: SVM (Hiperparametrizado con Kappa de Cohen)\n")
        f.write(f"Kappa de Cohen: {best_kappa}\n")
        f.write(f"Accuracy: {best_accuracy}\n")
        f.write(f"Precision: {best_precision}\n")
        f.write(f"Recall: {best_recall}\n")
        f.write(f"F1-score: {best_f1}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")
        f.write(f"Mejores Hiperparámetros: {grid_search.best_params_}\n")

    return best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1

exclude_cols = ['lrscale', 'lrscale2_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']

best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_chi2_minmax_hyper(df2strat, 'lrscale_fc', (7, 20), scale_columns=scale_cols, exclude_columns=exclude_cols, transform_negative=True)

print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

Fitting 2 folds for each of 12 candidates, totalling 24 fits
